# Imagegeneratie-applicaties bouwen (OpenAI)

Er is meer aan grote taalmodellen (LLM's) dan tekstgeneratie. Je kunt ook afbeeldingen genereren uit tekstbeschrijvingen. Afbeeldingen als modaliteit zijn nuttig in MedTech, architectuur, toerisme, game-ontwikkeling, marketing en meer. In deze les werken we met de huidige **GPT Image**-modellen op het OpenAI-platform.

## Leerdoelen

Aan het einde van deze les kun je:

- Uitleggen wat imagegeneratie is en waar het nuttig is.
- Begrijpen wat de `gpt-image` model familie is en hoe deze verschilt van de legacy DALL·E-modellen.
- Een imagegeneratie-applicatie bouwen en afbeeldingen bewerken.

## Wat is imagegeneratie?

Imagegeneratiemodellen maken afbeeldingen aan de hand van een tekstprompt. Moderne modellen zoals `gpt-image` leren tijdens training de relatie tussen tekst en afbeeldingen, en zetten vervolgens iteratief willekeurige ruis om in een afbeelding die bij je prompt past.

Twee bekende families van imagemodellen zijn:

- **`gpt-image` (OpenAI)** — de huidige generatie die in deze les wordt gebruikt. Het ondersteunt tekst-naar-afbeelding generatie en het bewerken van afbeeldingen (inpainting met een masker).
- **Midjourney** — een populair model van derden met een eigen service en Discord-gebaseerde workflow.

> De oudere OpenAI imaginemodellen — **DALL·E 2** en **DALL·E 3** — zijn legacy. DALL·E 3 is niet langer beschikbaar voor nieuwe implementaties, en functies zoals `create_variation` bestonden alleen bij DALL·E 2. Gebruik voor nieuwe toepassingen de `gpt-image` modellen.

> **Belangrijk:** `gpt-image` modellen geven de gegenereerde afbeelding terug als **base64** (`b64_json`), niet als een URL. Je code decodeert de base64-string naar bytes en slaat die op — er is geen afbeeldings-URL om te downloaden.


## Je eerste applicatie voor het genereren van afbeeldingen bouwen

Wat heb je nodig om een applicatie voor het genereren van afbeeldingen te bouwen? Je hebt de volgende bibliotheken nodig:

- **python-dotenv**, het wordt sterk aanbevolen om deze bibliotheek te gebruiken om je geheimen in een *.env* bestand gescheiden van de code te houden.
- **openai**, deze bibliotheek gebruik je om te communiceren met de OpenAI API.
- **pillow**, om met afbeeldingen te werken in Python.


1. Maak een bestand *.env* aan met de volgende inhoud:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Verzamel de bovenstaande bibliotheken in een bestand genaamd *requirements.txt*, zoals volgt:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Maak vervolgens een virtuele omgeving aan en installeer de bibliotheken:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Voor Windows, gebruik de volgende opdrachten om je virtuele omgeving aan te maken en te activeren:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Voeg de volgende code toe in het bestand genaamd *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Maak een OpenAI-object aan (leest OPENAI_API_KEY uit je .env)
    client = openai.OpenAI()


    try:
        # Maak een afbeelding met behulp van de afbeeldingsgeneratie-API
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Voer hier je prompttekst in
            size='1024x1024',
            n=1
        )
        # Stel de map in voor de opgeslagen afbeelding
        image_dir = os.path.join(os.curdir, 'images')

        # Als de map niet bestaat, maak deze aan
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialiseer het afbeeldingspad (let op, het bestandstype moet png zijn)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image modellen geven de afbeelding als base64 terug (b64_json), niet als URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Toon de afbeelding in de standaard afbeeldingsviewer
        image = Image.open(image_path)
        image.show()

    # vang uitzonderingen op
    except openai.BadRequestError as err:
        print(err)

    ```

Laten we deze code uitleggen:

- Eerst importeren we de benodigde libraries, waaronder de OpenAI library, de dotenv library, de base64 module en de Pillow library.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Daarna maken we de client aan, die de API-sleutel leest uit je ``.env``.

    ```python
    # Maak OpenAI-object aan
    client = openai.OpenAI()
    ```

- Vervolgens genereren we de afbeelding:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Voer hier uw prompttekst in
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` modellen geven de afbeelding terug als een **base64** string in `data[0].b64_json`. We decoderen dit naar bytes en schrijven het naar een bestand — er is geen URL om te downloaden.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Tenslotte openen we de afbeelding en gebruiken de standaard afbeeldingsviewer om deze te tonen:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Meer details over het genereren van de afbeelding

Laten we kijken naar de parameters van `images.generate`:

- **model** is het afbeeldingsmodel dat gebruikt wordt (bijvoorbeeld `gpt-image-1`).
- **prompt** is de tekstprompt die gebruikt wordt om de afbeelding te genereren. Hier is dat "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size** is de grootte van de gegenereerde afbeelding (bijvoorbeeld `1024x1024`, `1536x1024`, `1024x1536` of `"auto"`).
- **n** is het aantal gegenereerde afbeeldingen. Hier genereren we er één.

> Beeldmodellen gebruiken geen `temperature` parameter — dat is een controle voor tekstgeneratie. Voor meer variatie roep je de API opnieuw aan; voor minder variatie maak je je prompt specifieker.

## Extra mogelijkheden van afbeeldingsgeneratie

Je hebt gezien hoe je met een paar regels Python een afbeelding genereert. `gpt-image` modellen kunnen ook een bestaande afbeelding **bewerken**. Door een afbeelding, een optioneel **masker** (dat het te wijzigen gebied markeert) en een prompt mee te geven, kun je een deel van een afbeelding wijzigen — bijvoorbeeld een hoed toevoegen aan onze konijn.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# bewerkingen worden ook geretourneerd als base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

De basisafbeelding bevat alleen het konijn; de uiteindelijke afbeelding heeft de hoed op het konijn.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
